# Data Processing

## Processing 

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from ifood.data_processing.offers import read_offer_json, get_offer_channels
from ifood.data_processing.profile import read_profile_json, remove_profile_fillna, get_account_age
from ifood.data_processing.transactions import read_transactions_json, get_transaction_id, read_transactions_array
from ifood.data_processing.offer_lifecycle import build_offer_lifecycle
from ifood.data_processing.modeling import select_offer, select_non_offer, get_rows, get_target, join_all_dataframes, get_account_split

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

from pathlib import Path

spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/28 23:34:19 WARN Utils: Your hostname, MacBook-Air-de-Vitor.local, resolves to a loopback address: 127.0.0.1; using 192.168.3.49 instead (on interface en0)
26/05/28 23:34:19 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/28 23:34:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
data_path = Path.cwd().parent / 'data' / 'raw'

raw_offers = read_offer_json(spark, data_path, "offers.json")
raw_profile = read_profile_json(spark, data_path, "profile.json")
raw_transactions = read_transactions_json(spark, data_path, "transactions.json")

offers = (
    raw_offers
    .transform(get_offer_channels)
)

profile = (
    raw_profile
    .transform(remove_profile_fillna)
    .transform(get_account_age)
)

transactions = (
    raw_transactions
    .transform(read_transactions_array)
    .transform(get_transaction_id)
)

lifecycle = build_offer_lifecycle(transactions, offers)

In [ ]:
# Save lifecycle because it iterates over the rows and it takes time
lifecycle.write.mode("overwrite").parquet(str(Path.cwd().parent / 'data' / 'processed'/ 'lifecycle'))

In [6]:
lifecycle = spark.read.parquet(str(Path.cwd().parent / 'data' / 'processed'/ 'lifecycle'))

In [7]:
selected_offer = select_offer(lifecycle)
non_offer = select_non_offer(transactions)

rows = get_rows(selected_offer, non_offer)
target = get_target(rows, transactions)

df = (
    join_all_dataframes(rows, target, transactions, profile, offers)
    .transform(get_account_split)
)

In [8]:
df.write.mode("overwrite").parquet(str(Path.cwd().parent / 'data' / 'processed'/ 'unified'))

## Appendix 

### Offers

In [0]:
raw_offers.orderBy("offer_type", "channels").display()

In [0]:
display(
    raw_offers
    .withColumn("web_channel", F.array_contains(F.col("channels"), "web").cast("int"))
    .withColumn("mobile_channel", F.array_contains(F.col("channels"), "mobile").cast("int"))
    .withColumn("email_channel", F.array_contains(F.col("channels"), "email").cast("int"))
    .withColumn("social_channel", F.array_contains(F.col("channels"), "social").cast("int"))
    .drop("channels")
)

### Profile

In [0]:
display(raw_profile)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
display(
    raw_profile
    .agg(
        F.min(F.col("registered_on")),
        F.max(F.col("registered_on")),
    )
)

In [0]:
display(
    profile
)

Databricks visualization. Run in Databricks to view.

In [0]:
raw_profile.groupBy("age").count().orderBy("age").display()

Databricks visualization. Run in Databricks to view.

In [0]:
(
    raw_profile
    .groupBy(
        F.col("age") == 118,
        F.col("credit_card_limit").isNull(),
        F.col("gender").isNull()
    )
    .count()
).show()

In [0]:
profile.groupby("gender").count().display()

### Transactions

In [0]:
display(transactions.orderBy(F.rand()))

Databricks visualization. Run in Databricks to view.

In [0]:
display(
    transactions
    .groupBy("account_id")
    .agg(
        F.count("*"),
        F.min("time_since_test_start"),
        F.max("time_since_test_start")
    )
)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# All offers are in transactions 
display(
    transactions
    .where(F.col("offer_id").isNotNull())
    .select(F.col("offer_id"), F.lit(1).alias("in_transactions"))
    .join(
        offers
        .select(
            F.col("offer_id"), F.lit(1).alias("in_offers")
        ), 
    ['offer_id'], "full")
    .distinct()
    .agg(
        F.count("*"),
        F.sum("in_transactions"),
        F.sum("in_offers")
    )
)

In [0]:
# All accounts are in transactions
display(
    transactions
    .where(F.col("account_id").isNotNull())
    .select(F.col("account_id"), F.lit(1).alias("in_transactions"))
    .join(
        profile
        .select(
            F.col("account_id"), F.lit(1).alias("in_profile")
        ), 
    ['account_id'], "full")
    .distinct()
    .agg(
        F.count("*"),
        F.sum("in_transactions"),
        F.sum("in_profile")
    )
)

### Questions about Transactions

#### How many offers does each customer receive?

In [0]:
display(
    transactions
    .where(F.col("event")=="offer received")
    .groupBy("account_id")
    .agg(F.count("*").alias("n_offer"))
    .groupBy("n_offer")
    .count()
    .orderBy("n_offer")
)

Databricks visualization. Run in Databricks to view.

#### How many customers receive multiple offers simultaneously?

In [0]:
offer_end_rows = (
    transactions
    .where(F.col("event") == "offer received")
    .join(offers.select("offer_id", "duration"), "offer_id", "left")
    .withColumn("event", F.lit("offer end"))
    .withColumn("time_since_test_start", F.col("time_since_test_start") + F.col("duration"))
    .drop("duration")
)

transactions_with_offer_end = transactions.unionByName(offer_end_rows)

days = spark.createDataFrame([(i,) for i in range(30)], ["time_since_test_start"])
all_days = profile.select("account_id").crossJoin(days)

w = Window.partitionBy("account_id").orderBy("time_since_test_start")

display(
    transactions_with_offer_end
    .withColumn("time_since_test_start", F.floor(F.col("time_since_test_start")))
    .join(all_days, ["account_id", "time_since_test_start"], "full")

    .groupBy("account_id", "time_since_test_start")
    .agg(
        F.sum(F.when(F.col("event")=="offer received", 1).otherwise(0)).alias("offer_received"),
        F.sum(F.when(F.col("event").isin("offer end", "offer_reward"), 1).otherwise(0)).alias("offer_end")
    )
    .withColumn("n_offers", F.sum("offer_received").over(w) - F.sum("offer_end").over(w))

    .groupBy("account_id")
    .agg(F.max("n_offers").alias("max_n_offers"))

    .groupBy("max_n_offers")
    .count()
)

Databricks visualization. Run in Databricks to view.

#### In the offer lifecycle, how many 'offer viewed' and 'offer completed' events are missing from the lifecycle table? Are there multiple views or completions per offer?

In [0]:
loss_viewed = (
    transactions
    .where(F.col("event").isin("offer viewed"))
    .join(lifecycle.select("view_transaction_id"), transactions.transaction_id == lifecycle.view_transaction_id, "leftanti")
).count()

loss_completed = (
    transactions
    .where(F.col("event").isin("offer completed"))
    .withColumn("transaction_id", F.col("transaction_id").cast("string"))
    .join(lifecycle.select("comp_transaction_id"), transactions.transaction_id == lifecycle.comp_transaction_id, "leftanti")
).count()

display(
    transactions
    .where(F.col("event")!="transaction")
    .groupBy("event")
    .count()
    .withColumn(
        "loss_transactions",
        F.when(F.col("event") == "offer viewed", F.lit(loss_viewed))
         .when(F.col("event") == "offer completed", F.lit(loss_completed))
         .otherwise(F.lit(0))
    )
    .withColumn("pct", F.col("loss_transactions") / F.lit(76277))
)

In [0]:
display(
    lifecycle
    .groupBy(
        F.col("received_at").isNull(),
        F.col("viewed_at").isNull(),
        F.col("completed_at").isNull()
    )
    .count()
    .withColumn("pct", F.col("count") / F.lit(76277))
    .withColumn("right_pct", F.col("pct") * F.lit(9.8e-1))
)

#### How many offers are single offer?

In [0]:
overlap_window = Window.partitionBy("account_id").orderBy("received_at")

lifecycle_period = (
    lifecycle
    .withColumn("period_end", F.least(F.col("ended_at"), F.col("completed_at")))
)

overlap_count = (
    lifecycle_period.alias("a")
    .join(
        lifecycle_period.alias("b"),
        (F.col("a.account_id") == F.col("b.account_id")) &
        (F.col("a.received_at") < F.col("b.period_end")) &
        (F.col("a.period_end") > F.col("b.received_at")),
        "left"
    )
    .where(F.col("a.transaction_id") != F.col("b.transaction_id"))
    .groupBy("a.account_id", "a.transaction_id")
    .agg(F.count("b.offer_id").alias("n_overlapping_offers"))
)

display(
    lifecycle
    .join(overlap_count, ["account_id", "transaction_id"], "left")
    .withColumn("n_overlapping_offers", F.coalesce(F.col("n_overlapping_offers"), F.lit(0)))
    .groupBy("n_overlapping_offers")
    .count()
    .withColumn("pct", F.round(F.col("count") / F.lit(77452),4))
    .orderBy("n_overlapping_offers")
)

Databricks visualization. Run in Databricks to view.

#### Distribution of Days Between Offer Completion and Next Offer Viewed

In [0]:
w = Window.partitionBy("account_id").orderBy("received_at")

display(
    lifecycle
    .withColumn("next_viewed_at", F.lead("viewed_at", 1).over(w))
    .withColumn("offer_ended_at", F.least(F.col("ended_at"), F.col("completed_at"), F.lit(29)))
    .withColumn("days_between", F.coalesce(F.col("next_viewed_at"), F.lit(29)) - F.col("offer_ended_at"))
    .join(overlap_count, "transaction_id", "left_anti")
    .select("transaction_id", "received_at", "ended_at", "completed_at", "offer_ended_at", "next_viewed_at", "days_between")
    .groupBy("days_between")
    .count()
    .withColumn("cumsum", F.sum("count").over(Window.orderBy(F.col("days_between").desc())))
    .withColumn("ecdf_mirror", F.col("cumsum")/F.lit(77452))
)

Databricks visualization. Run in Databricks to view.